# BugMine — Results Analysis

Load `results/runs.jsonl`, compute extraction / validation / generalization metrics, and refresh `RESULTS.md`.

In [ ]:
import json
from pathlib import Path

ROOT = Path('..').resolve()
RUNS = ROOT / 'results' / 'runs.jsonl'

records = []
if RUNS.exists():
    for line in RUNS.read_text(encoding='utf-8').splitlines():
        if line.strip():
            records.append(json.loads(line))

len(records)

In [ ]:
def rate(n, d):
    return f"{100 * n / d:.1f}%" if d else '—'

attempted = len(records)
extracted = sum(1 for r in records if r.get('extracted'))
compiled = sum(1 for r in records if r.get('codegen', {}).get('compiled'))
codegen_failed = sum(1 for r in records if r.get('codegen') and not r['codegen'].get('compiled'))

with_validation = [r for r in records if r.get('validation') and r.get('codegen', {}).get('compiled')]
self_validated = sum(1 for r in with_validation if r['validation']['verdict'] == 'valid')
too_weak = sum(1 for r in with_validation if 'too weak' in r['validation'].get('notes', '').lower())
over_constrained = sum(1 for r in with_validation if 'over-constrained' in r['validation'].get('notes', '').lower())
inconclusive = sum(1 for r in with_validation if r['validation']['verdict'] == 'inconclusive')

generalization_hits = [
    (r['finding_id'], len(r.get('generalization', {}).get('hits', [])))
    for r in records if r.get('generalization')
]

metrics = {
    'attempted': attempted,
    'extracted': extracted,
    'compiled': compiled,
    'codegen_failed': codegen_failed,
    'self_validated': self_validated,
    'too_weak': too_weak,
    'over_constrained': over_constrained,
    'inconclusive': inconclusive,
    'generalization_hits': generalization_hits,
}
metrics

In [ ]:
try:
    import matplotlib.pyplot as plt

    labels = ['Self-validated', 'Too weak', 'Over-constrained', 'Inconclusive']
    values = [self_validated, too_weak, over_constrained, inconclusive]
    if any(values):
        plt.figure(figsize=(6, 4))
        plt.bar(labels, values)
        plt.title('Self-validation outcomes (compiled tests only)')
        plt.ylabel('Findings')
        plt.tight_layout()
        plt.show()
except ImportError:
    print('matplotlib not installed — skipping plot')

In [ ]:
def md_table(headers, rows):
    lines = ['| ' + ' | '.join(headers) + ' |', '|' + '|'.join(['---'] * len(headers)) + '|']
    for row in rows:
        lines.append('| ' + ' | '.join(str(c) for c in row) + ' |')
    return '\n'.join(lines)

extraction_rows = [
    ['Findings attempted', attempted, '—'],
    ['Specs extracted', extracted, rate(extracted, attempted)],
    ['Tests compiled', compiled, rate(compiled, attempted)],
    ['Codegen failed', codegen_failed, rate(codegen_failed, attempted)],
]

validation_rows = [
    ['Findings with compiled tests', len(with_validation), '—'],
    ['Self-validated', self_validated, rate(self_validated, len(with_validation))],
    ['Too weak', too_weak, rate(too_weak, len(with_validation))],
    ['Over-constrained', over_constrained, rate(over_constrained, len(with_validation))],
    ['Inconclusive', inconclusive, rate(inconclusive, len(with_validation))],
]

gen_rows = [[fid, hits, ''] for fid, hits in generalization_hits] or [['_none yet_', '—', '—']]

detail_rows = []
for r in records:
    cat = r.get('extracted', {}).get('category', '—')
    compiled_ok = 'yes' if r.get('codegen', {}).get('compiled') else 'no'
    verdict = r.get('validation', {}).get('verdict', '—')
    hits = len(r.get('generalization', {}).get('hits', []))
    detail_rows.append([r['finding_id'], cat, 'yes' if r.get('extracted') else 'no', compiled_ok, verdict, hits or '—'])

results_md = f'''# BugMine Results

_Auto-generated from `results/runs.jsonl` via `notebooks/analysis.ipynb`._

## Extraction success rate

{md_table(['Metric', 'Count', 'Rate'], extraction_rows)}

## Self-validation rate

{md_table(['Metric', 'Count', 'Rate'], validation_rows)}

## Generalization

{md_table(['Finding ID', 'Hits', 'Notes'], gen_rows)}

## Limitations & failure modes

_Document extraction errors, Halmos timeouts, property mismatch, and false generalization hits._

## Per-finding detail

{md_table(['Finding ID', 'Category', 'Extracted', 'Compiled', 'Self-validates', 'Generalizes'], detail_rows or [['_none yet_', '—', '—', '—', '—', '—']])}
'''

out = ROOT / 'RESULTS.md'
out.write_text(results_md, encoding='utf-8')
print(f'Wrote {out}')